# 面试问题：Agent 代表用户调用远程工具时，OAuth 委托授权怎样避免 confused deputy？

**回答主线。** Agent 不应拿用户的万能 bearer token 到处转发。每个下游资源需要明确 audience/resource、tenant、最小 scope、过期时间和主体；跨资源调用通过 token exchange/新授权获得更窄 token。高风险动作还要 step-up approval，DPoP 类 proof 可把 token 绑定到客户端密钥，并用 jti/nonce 防重放。

下面实现 claims、精确 audience 验证、protected-resource discovery、教学版 proof、重放缓存、scope narrowing、动作审批和撤销审计。密码学部分只模拟“绑定字段”语义，不是生产 JWT/签名实现。


In [ ]:
import hashlib, hmac, json, time  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。

# 使用固定时间建立可复现 token 生命周期。
NOW161 = 2_000_000_000  # 计算并保存当前步骤的中间状态。
assert NOW161 > 0  # 用受控断言验证关键不变量。
assert len(hashlib.sha256(b"x").hexdigest()) == 64  # 用受控断言验证关键不变量。
assert hmac.compare_digest("a", "a")  # 用受控断言验证关键不变量。


## 1. Access token claims 绑定 issuer、subject、tenant、audience 与 scope

Audience 必须是精确资源标识，不能用域名后缀或字符串包含判断。Scope 表示允许动作上限，资源服务器还要结合对象 ACL 再授权。


In [ ]:
# Claims 是资源服务器执行授权判定所需的最小结构化输入。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class TokenClaims161:  # 定义承载本节状态与行为的数据结构。
    issuer: str  # 执行当前语句以推进本节示例。
    subject: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    audience: str  # 执行当前语句以推进本节示例。
    scopes: tuple  # 执行当前语句以推进本节示例。
    expires_at: int  # 执行当前语句以推进本节示例。
    jti: str  # 执行当前语句以推进本节示例。
    confirmation_key: str  # 执行当前语句以推进本节示例。

token161 = TokenClaims161("https://auth.example", "user-7", "tenant-a", "https://mail.example", ("mail.read",), NOW161 + 300, "jti-1", "key-1")  # 计算并保存当前步骤的中间状态。
assert token161.audience == "https://mail.example"  # 用受控断言验证关键不变量。
assert token161.scopes == ("mail.read",)  # 用受控断言验证关键不变量。
assert token161.expires_at > NOW161  # 用受控断言验证关键不变量。


## 2. Resource server 本地验证所有关键 claims

验签之外还要校验 issuer allowlist、精确 aud、过期、tenant 和所需 scope。缺 scope 返回结构化 insufficient_scope，引导重新授权，而不是让模型猜测更高权限 token。


In [ ]:
def authorize161(token, expected_issuer, resource, tenant, required_scopes, now):  # 定义本节可复用的核心函数。
    # 返回明确错误列表，任何一项失败都拒绝执行工具。
    errors = []  # 计算并保存当前步骤的中间状态。
    if token.issuer != expected_issuer: errors.append("invalid_issuer")  # 按当前条件选择后续控制路径。
    if token.audience != resource: errors.append("invalid_audience")  # 按当前条件选择后续控制路径。
    if token.tenant != tenant: errors.append("tenant_mismatch")  # 按当前条件选择后续控制路径。
    if token.expires_at <= now: errors.append("expired")  # 按当前条件选择后续控制路径。
    if not set(required_scopes).issubset(token.scopes): errors.append("insufficient_scope")  # 按当前条件选择后续控制路径。
    return errors  # 返回当前分支计算出的结果。

assert authorize161(token161, "https://auth.example", "https://mail.example", "tenant-a", {"mail.read"}, NOW161) == []  # 用受控断言验证关键不变量。
assert "invalid_audience" in authorize161(token161, "https://auth.example", "https://drive.example", "tenant-a", set(), NOW161)  # 用受控断言验证关键不变量。
assert "insufficient_scope" in authorize161(token161, "https://auth.example", "https://mail.example", "tenant-a", {"mail.send"}, NOW161)  # 用受控断言验证关键不变量。


## 3. Protected Resource Metadata discovery 也需要 SSRF 门禁

Client 从受保护资源发现 authorization server，但不能让任意 URL 指向内网或泄露 token。这里只允许 HTTPS、精确 host allowlist，metadata issuer 也必须匹配授权策略。


In [ ]:
from urllib.parse import urlparse  # 导入本单元所需的依赖。

def validate_resource_metadata161(resource_url, metadata, allowed_hosts):  # 定义本节可复用的核心函数。
    # 不做字符串后缀匹配，解析后的 hostname 必须精确命中 allowlist。
    parsed = urlparse(resource_url)  # 计算并保存当前步骤的中间状态。
    if parsed.scheme != "https" or parsed.hostname not in allowed_hosts or parsed.username or parsed.password:  # 按当前条件选择后续控制路径。
        return False  # 返回当前分支计算出的结果。
    auth_servers = metadata.get("authorization_servers", [])  # 计算并保存当前步骤的中间状态。
    return bool(auth_servers) and all(urlparse(url).scheme == "https" for url in auth_servers)  # 返回当前分支计算出的结果。

metadata161 = {"authorization_servers": ["https://auth.example"]}  # 计算并保存当前步骤的中间状态。
assert validate_resource_metadata161("https://mail.example/.well-known/oauth-protected-resource", metadata161, {"mail.example"})  # 用受控断言验证关键不变量。
assert not validate_resource_metadata161("https://mail.example.evil.test/meta", metadata161, {"mail.example"})  # 用受控断言验证关键不变量。
assert not validate_resource_metadata161("http://mail.example/meta", metadata161, {"mail.example"})  # 用受控断言验证关键不变量。


## 4. Proof-of-possession 绑定 method、URL、token hash、jti 与 nonce

真实 DPoP 使用非对称密钥和签名 JWT。本教学版用 HMAC 模拟待签字段与验证顺序，只演示为什么偷到 access token 仍缺少绑定密钥，以及 method/URL 改变会使 proof 失效。


In [ ]:
def make_proof161(method, url, access_token, proof_jti, nonce, secret):  # 定义本节可复用的核心函数。
    # canonical payload 显式绑定 HTTP 语义与 access token 摘要。
    ath = hashlib.sha256(access_token.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
    payload = json.dumps({"htm": method.upper(), "htu": url, "ath": ath, "jti": proof_jti, "nonce": nonce}, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    signature = hmac.new(secret, payload.encode(), hashlib.sha256).hexdigest()  # 计算并保存当前步骤的中间状态。
    return payload, signature  # 返回当前分支计算出的结果。

def verify_proof161(payload, signature, secret, method, url, access_token, nonce):  # 定义本节可复用的核心函数。
    expected_payload, expected_signature = make_proof161(method, url, access_token, json.loads(payload)["jti"], nonce, secret)  # 计算并保存当前步骤的中间状态。
    return payload == expected_payload and hmac.compare_digest(signature, expected_signature)  # 返回当前分支计算出的结果。

proof_payload161, proof_sig161 = make_proof161("POST", "https://mail.example/send", "opaque-token", "proof-1", "nonce-7", b"client-key")  # 计算并保存当前步骤的中间状态。
assert verify_proof161(proof_payload161, proof_sig161, b"client-key", "POST", "https://mail.example/send", "opaque-token", "nonce-7")  # 用受控断言验证关键不变量。
assert not verify_proof161(proof_payload161, proof_sig161, b"client-key", "GET", "https://mail.example/send", "opaque-token", "nonce-7")  # 用受控断言验证关键不变量。
assert not verify_proof161(proof_payload161, proof_sig161, b"wrong", "POST", "https://mail.example/send", "opaque-token", "nonce-7")  # 用受控断言验证关键不变量。


## 5. jti/nonce replay cache 在执行前原子消费

有效签名可以被复制；资源服务器需在有效窗口内拒绝重复 proof jti。检查与写入必须原子，否则并发请求都可能通过。nonce 还能由服务端推动 proof 新鲜度。


In [ ]:
class ReplayCache161:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.seen = {}  # 定义本节可复用的核心函数。

    def consume(self, issuer, proof_jti, expires_at, now):  # 定义本节可复用的核心函数。
        # 先清理过期项，再以 issuer+jti 作为隔离键原子登记。
        self.seen = {k: v for k, v in self.seen.items() if v > now}  # 计算并保存当前步骤的中间状态。
        key = (issuer, proof_jti)  # 计算并保存当前步骤的中间状态。
        if key in self.seen: return False  # 按当前条件选择后续控制路径。
        self.seen[key] = expires_at  # 计算并保存当前步骤的中间状态。
        return True  # 返回当前分支计算出的结果。

replay161 = ReplayCache161()  # 计算并保存当前步骤的中间状态。
assert replay161.consume("https://auth.example", "proof-1", NOW161 + 60, NOW161)  # 用受控断言验证关键不变量。
assert not replay161.consume("https://auth.example", "proof-1", NOW161 + 60, NOW161 + 1)  # 用受控断言验证关键不变量。
assert replay161.consume("https://other-auth.example", "proof-1", NOW161 + 60, NOW161 + 1)  # 用受控断言验证关键不变量。


## 6. Token exchange 只能缩窄 scope 并更换目标 audience

上游 Agent token 不能直接交给下游工具。Exchange 验证主体/tenant 后签发面向目标 resource 的短期 token，requested scope 必须是上游 scope 子集；这样 drive token 不能被 mail 服务接受。


In [ ]:
def exchange_token161(parent, target_resource, requested_scopes, now, new_jti):  # 定义本节可复用的核心函数。
    # 交换只允许权限收缩，不允许延长超过父 token 的生命周期。
    requested_scopes = set(requested_scopes)  # 计算并保存当前步骤的中间状态。
    if not requested_scopes.issubset(parent.scopes):  # 按当前条件选择后续控制路径。
        raise PermissionError("scope escalation")  # 遇到非法合同立即显式失败。
    return TokenClaims161(parent.issuer, parent.subject, parent.tenant, target_resource, tuple(sorted(requested_scopes)), min(parent.expires_at, now + 120), new_jti, parent.confirmation_key)  # 返回当前分支计算出的结果。

broad161 = TokenClaims161(token161.issuer, token161.subject, token161.tenant, "https://agent.example", ("mail.read", "mail.send"), NOW161 + 300, "parent", "key-1")  # 计算并保存当前步骤的中间状态。
child161 = exchange_token161(broad161, "https://mail.example", {"mail.read"}, NOW161, "child")  # 计算并保存当前步骤的中间状态。
assert child161.audience == "https://mail.example"  # 用受控断言验证关键不变量。
assert child161.scopes == ("mail.read",) and child161.expires_at == NOW161 + 120  # 用受控断言验证关键不变量。
assert authorize161(child161, token161.issuer, "https://drive.example", token161.tenant, set(), NOW161) == ["invalid_audience"]  # 用受控断言验证关键不变量。


## 7. 高风险动作需要与参数绑定的 step-up approval

`mail.send`、付款和删除不能只依赖宽 scope。审批票据摘要覆盖 subject、resource、tool、canonical args 和过期时间；参数变化后旧票据立即失效。


In [ ]:
def action_digest161(subject, resource, tool, arguments):  # 定义本节可复用的核心函数。
    # canonical 参数保证审批绑定的是精确动作，而非模糊工具名。
    payload = json.dumps({"subject": subject, "resource": resource, "tool": tool, "args": arguments}, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload.encode()).hexdigest()  # 返回当前分支计算出的结果。

approval161 = {"digest": action_digest161("user-7", "https://mail.example", "mail.send", {"to": "a@example.com", "body": "hi"}), "expires_at": NOW161 + 30}  # 计算并保存当前步骤的中间状态。
same_action161 = action_digest161("user-7", "https://mail.example", "mail.send", {"body": "hi", "to": "a@example.com"})  # 计算并保存当前步骤的中间状态。
changed_action161 = action_digest161("user-7", "https://mail.example", "mail.send", {"to": "b@example.com", "body": "hi"})  # 计算并保存当前步骤的中间状态。
assert approval161["digest"] == same_action161  # 用受控断言验证关键不变量。
assert approval161["digest"] != changed_action161  # 用受控断言验证关键不变量。
assert approval161["expires_at"] > NOW161  # 用受控断言验证关键不变量。


## 8. 撤销、审计与 deny-by-default 属于执行路径

短期 token 仍需支持用户撤销会话、client key 或 grant。审计记录保存决策输入的摘要而非明文 token；授权服务不可用时，高风险写操作默认拒绝，低风险缓存读是否降级由策略决定。


In [ ]:
def authorize_with_revocation161(token, revoked_jtis, required_scopes, now):  # 定义本节可复用的核心函数。
    # 先查撤销，再复用完整 claims 校验；日志只记录 jti 与决策原因。
    if token.jti in revoked_jtis:  # 按当前条件选择后续控制路径。
        return False, {"jti": token.jti, "reason": "revoked"}  # 返回当前分支计算出的结果。
    errors = authorize161(token, "https://auth.example", token.audience, token.tenant, required_scopes, now)  # 计算并保存当前步骤的中间状态。
    return not errors, {"jti": token.jti, "reason": "ok" if not errors else errors[0]}  # 返回当前分支计算出的结果。

allowed161, audit_ok161 = authorize_with_revocation161(child161, set(), {"mail.read"}, NOW161)  # 计算并保存当前步骤的中间状态。
denied161, audit_bad161 = authorize_with_revocation161(child161, {"child"}, {"mail.read"}, NOW161)  # 计算并保存当前步骤的中间状态。
assert allowed161 and audit_ok161["reason"] == "ok"  # 用受控断言验证关键不变量。
assert not denied161 and audit_bad161["reason"] == "revoked"  # 用受控断言验证关键不变量。
assert "opaque-token" not in json.dumps(audit_ok161)  # 用受控断言验证关键不变量。


## 面试总结

- Agent 持有的是代表用户的委托权限，不是用户万能凭据；token 精确绑定 issuer、subject、tenant、audience、scope 和 expiry。
- 跨资源通过 exchange/重新授权获得更窄 token，资源服务器本地再做对象 ACL，防 confused deputy。
- DPoP 类 proof 绑定客户端密钥、method、URL、token hash、jti/nonce；高风险动作再绑定人类审批摘要。
- 生产使用标准 OAuth/JWT/密码库，本 Notebook 的 HMAC 只解释字段语义。

延伸阅读：[MCP Authorization Specification](https://modelcontextprotocol.io/specification/2025-06-18/basic/authorization)、[RFC 8707 Resource Indicators](https://www.rfc-editor.org/info/rfc8707/)、[RFC 9449 DPoP](https://www.ietf.org/rfc/rfc9449.html)。
